In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import tensorflow as tf

from deepsphere import HealpyGCNN
from deepsphere.healpy_layers import Healpy_ViT, Healpy_Transformer, HealpyPseudoConv

from msfm.utils import files

from deep_lss.models.delta_model import DeltaLossModel
from deep_lss.nets import resnet, transformer 
# from deep_lss.models.grid_model import GridLossModel
# from msfm.fiducial_pipeline import FiducialPipeline

24-03-21 06:18:13   imports.py INF   Setting up healpy to run on 32 CPUs 


In [3]:
physical_devices = tf.config.list_physical_devices("GPU")
for device in physical_devices:
    if device.device_type == "GPU":
        tf.config.experimental.set_memory_growth(device, True)

### constants

In [4]:
n_side = 512
data_vec_pix, _, _, _ = files.load_pixel_file()
n_z_bins = 8
n_second_to_last_channels = None
n_output = 7

24-03-21 06:18:15     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_512.h5 


### random batch

In [5]:
# for delta loss
local_batch_size = 4 * (2 * n_output + 1)
print(f"batch_size = {local_batch_size}")
batch = tf.random.uniform((local_batch_size, len(data_vec_pix), n_z_bins), dtype=tf.float32)

batch_size = 60


# ResNet baseline

In [6]:
layers = resnet.ResNetLayers(
    n_output,
    base_channels=32,
    downsampling_layers=3,
    cheby_layers=2,
    residual_layers=5,
    poly_degree=5,
    second_to_last_features=n_second_to_last_channels,
).get_layers()

resnet_model = DeltaLossModel(
    network=layers,
    n_side=n_side,
    indices=data_vec_pix,
    n_neighbors=20,
    input_shape=(None, len(data_vec_pix), n_z_bins),
)

resnet_model.tf_call(batch);

24-03-21 06:18:16    resnet.py WAR   No smoothing layer is included in the network 
24-03-21 06:18:16 regression_h INF   Using a dense regression head 
24-03-21 06:18:16 base_model.p INF   Initializing with a HealpyGCNN model 
Detected a reduction factor of 32.0, the input with nside 512 will be transformed to 16 during a forward pass. Checking for consistency with indices...
indices seem consistent...
Model: "healpy_gcnn_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 healpy_pseudo_conv (HealpyP  (None, 116224, 32)       1056      
 seudoConv)                                                      
                                                                 
 healpy_pseudo_conv_1 (Healp  (None, 29056, 64)        8256      
 yPseudoConv)                                                    
                                                                 
 healpy_pseudo_conv_2 (Healp  (None, 7264, 

In [7]:
%%timeit
resnet_model.tf_call(batch)

181 ms ± 25.1 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


# ViT

In [19]:
layers = transformer.ViTLayers(
    n_output,
    base_channels=None,
    downsampling_layers=None,
    hidden_dim=256,
    healpix_patch_fac=5,
    attention_heads=8,
    transformer_layers=8,
    pos_encoding=True,
    second_to_last_features=n_second_to_last_channels,
).get_layers()

# layers = [
#     Healpy_ViT(p=4, key_dim=128, num_heads=4, positional_encoding=True, n_layers=4, layer_norm=True),
#     tf.keras.layers.Flatten(),
#     tf.keras.layers.LayerNormalization(axis=-1),
#     # tf.keras.layers.Dense(n_second_to_last_channels, activation="relu"),
#     # tf.keras.layers.Dropout(0.0),
#     tf.keras.layers.Dense(n_output),
# ]

vit_model = DeltaLossModel(
    network=layers,
    n_side=n_side,
    indices=data_vec_pix,
    n_neighbors=20,
    input_shape=(None, len(data_vec_pix), n_z_bins),
)

vit_model.tf_call(batch);

Every patch consists of 1024 HEALPix pixels
24-03-21 06:32:10 regression_h INF   Using a dense regression head 
24-03-21 06:32:10 base_model.p INF   Initializing with a HealpyGCNN model 
Detected a reduction factor of 32.0, the input with nside 512 will be transformed to 16 during a forward pass. Checking for consistency with indices...
indices seem consistent...
Model: "healpy_gcnn_13"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 healpy__vi_t_5 (Healpy_ViT)  (None, 454, 2048)        152057856 
                                                                 
 flatten_6 (Flatten)         (None, 929792)            0         
                                                                 
 layer_normalization_118 (La  (None, 929792)           1859584   
 yerNormalization)                                               
                                                                 
 dense_206 (Dens

In [20]:
%%timeit
vit_model.tf_call(batch)

179 ms ± 16.9 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [9]:
# layers = [
#     Healpy_ViT(p=3, key_dim=128, num_heads=4, positional_encoding=True, n_layers=4, layer_norm=True),
#     tf.keras.layers.Flatten(),
#     tf.keras.layers.LayerNormalization(axis=-1),
#     # tf.keras.layers.Dense(n_second_to_last_channels, activation="relu"),
#     # tf.keras.layers.Dropout(0.0),
#     tf.keras.layers.Dense(n_output),
# ]

# vit_model = DeltaLossModel(
#     network=layers,
#     n_side=n_side,
#     indices=data_vec_pix,
#     n_neighbors=20,
#     input_shape=(None, len(data_vec_pix), n_z_bins),
# )

# Graph Transformer

In [23]:
n_base_channels = 32
n_downsampling = 4
activation = tf.nn.relu
hidden_dim = 128
n_heads = 4
positional_encoding = True
n_layers = 4

layers = []

n_channels = n_base_channels
for _ in range(n_downsampling):
    layers.append(HealpyPseudoConv(p=1, Fout=n_channels, activation=activation))
    n_channels *= 2
    
layers.append(Healpy_Transformer(key_dim=hidden_dim, num_heads=n_heads, positional_encoding=True, n_layers=n_layers))

layers.append(tf.keras.layers.Flatten())
layers.append(tf.keras.layers.LayerNormalization(axis=-1))
layers.append(tf.keras.layers.Dense(n_second_to_last_channels, activation="relu"))
layers.append(tf.keras.layers.Dropout(0.0))
layers.append(tf.keras.layers.Dense(n_output))

gt_model = DeltaLossModel(
    network=layers,
    n_side=n_side,
    indices=data_vec_pix,
    n_neighbors=20,
    input_shape=(None, len(data_vec_pix), n_z_bins),
)

24-02-14 01:52:02 base_model.p INF   Initializing with a HealpyGCNN model 
Detected a reduction factor of 16.0, the input with nside 512 will be transformed to 32 during a forward pass. Checking for consistency with indices...
indices seem consistent...
Model: "healpy_gcnn_19"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 healpy_pseudo_conv_37 (Heal  (None, 116224, 32)       1056      
 pyPseudoConv)                                                   
                                                                 
 healpy_pseudo_conv_38 (Heal  (None, 29056, 64)        8256      
 pyPseudoConv)                                                   
                                                                 
 healpy_pseudo_conv_39 (Heal  (None, 7264, 128)        32896     
 pyPseudoConv)                                                   
                                                              

In [22]:
%%timeit
gt_model(batch)

235 ms ± 279 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
# layers = [
#     HealpyPseudoConv(p=1, Fout=8, activation=tf.nn.relu),
#     HealpyPseudoConv(p=1, Fout=16, activation=tf.nn.relu),
#     HealpyPseudoConv(p=1, Fout=32, activation=tf.nn.relu),
#     HealpyPseudoConv(p=1, Fout=64, activation=tf.nn.relu),
#     Healpy_Transformer(key_dim=64, num_heads=4, positional_encoding=True, n_layers=4),
#     tf.keras.layers.Flatten(),
#     tf.keras.layers.LayerNormalization(axis=-1),
#     tf.keras.layers.Dense(n_output)    
# ]

# layers = [
#     HealpyPseudoConv(p=1, Fout=16, activation=tf.nn.relu),
#     HealpyPseudoConv(p=1, Fout=32, activation=tf.nn.relu),
#     HealpyPseudoConv(p=1, Fout=64, activation=tf.nn.relu),
#     Healpy_Transformer(key_dim=64, num_heads=4, positional_encoding=True, n_layers=4),
#     tf.keras.layers.Flatten(),
#     tf.keras.layers.LayerNormalization(axis=-1),
#     tf.keras.layers.Dense(n_output)    
# ]


# gt_model = DeltaLossModel(
#     network=layers,
#     n_side=n_side,
#     indices=data_vec_pix,
#     n_neighbors=20,
#     input_shape=(None, len(data_vec_pix), n_z_bins),
# )